# Collaboration Network Growth Animation

Renders the history of a GitHub project as a **movie**: the collaboration network grows from a single contributor outward, frame by frame, one time period at a time.

**Two outputs:**
- `OUTPUT_FILE` — GIF or MP4 (dark background, glow nodes, suitable for sharing)
- Plotly figure — interactive version with a play button and time scrubber (runs in the notebook)

**Layout:** spring layout computed once on the final graph, so well-connected (core) contributors cluster at the centre and peripheral contributors radiate outward. Nodes are revealed progressively; new arrivals flash brightly on entry.

**Node colour** follows the `plasma` colourmap: early contributors are purple/blue, later arrivals are orange/yellow.

## Configuration

In [ ]:
REPO            = "facebook/react"
GITHUB_TOKEN    = ""
MAX_PRS         = 300     # merged PRs to analyse — more = longer movie
TIME_RESOLUTION = 'Q'     # 'M' monthly  'Q' quarterly  'Y' yearly
OUTPUT_FILE     = 'collaboration_growth.gif'  # .gif (Pillow) or .mp4 (ffmpeg)
FPS             = 4       # frames per second

In [ ]:
import time, requests
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, PillowWriter
from collections import defaultdict
from itertools import combinations
try:
    import plotly.graph_objects as go
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print('plotly not installed — skipping interactive output')

## Fetch PR data

In [ ]:
def make_headers(token=''):
    h = {"Accept": "application/vnd.github+json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    return h


def paginate(url, headers, params=None, max_pages=None):
    results, page = [], 1
    while True:
        p = {**(params or {}), 'page': page, 'per_page': 100}
        r = requests.get(url, headers=headers, params=p, timeout=30)
        if r.status_code == 403:
            raise PermissionError("Rate limit hit — set GITHUB_TOKEN.")
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        results.extend(data)
        if (max_pages and page >= max_pages) or 'next' not in r.links:
            break
        page += 1
    return results


def fetch_pr_data(repo, headers, max_prs=300):
    print(f"Fetching PRs from {repo}...")
    raw = paginate(
        f"https://api.github.com/repos/{repo}/pulls",
        headers, {"state": "closed"},
        max_pages=(max_prs // 100 + 1) if max_prs else None,
    )
    merged = [pr for pr in raw if pr.get('merged_at')][:max_prs]

    records = []
    for i, pr in enumerate(merged):
        num    = pr['number']
        author = (pr.get('user') or {}).get('login')
        date   = pd.to_datetime(pr['merged_at'])
        reviews = paginate(
            f"https://api.github.com/repos/{repo}/pulls/{num}/reviews", headers
        )
        reviewers = list(
            {(r.get('user') or {}).get('login') for r in reviews} - {None, author}
        )
        records.append({'pr': num, 'author': author, 'reviewers': reviewers, 'date': date})
        if (i + 1) % 50 == 0:
            print(f"  {i + 1}/{len(merged)} PRs")
        time.sleep(0.05)

    print(f"Done — {len(records)} merged PRs")
    return records

## Build temporal snapshots

In [ ]:
def build_snapshots(pr_records, freq='Q'):
    """Cumulative network state at each time period."""
    for pr in pr_records:
        pr['period'] = pr['date'].to_period(freq)
    periods = sorted({pr['period'] for pr in pr_records})

    cum_nodes = {}  # node  -> {'first_seen': period, 'pr_count': int}
    cum_edges = {}  # (a,b) -> {'first_seen': period, 'weight': int}
    snapshots = []

    for period in periods:
        for pr in (p for p in pr_records if p['period'] == period):
            participants = list({p for p in [pr['author']] + pr['reviewers'] if p})
            for p in participants:
                if p not in cum_nodes:
                    cum_nodes[p] = {'first_seen': period, 'pr_count': 0}
                cum_nodes[p]['pr_count'] += 1
            for a, b in combinations(participants, 2):
                key = tuple(sorted([a, b]))
                if key not in cum_edges:
                    cum_edges[key] = {'first_seen': period, 'weight': 0}
                cum_edges[key]['weight'] += 1

        snapshots.append({
            'period': period,
            'nodes':  dict(cum_nodes),
            'edges':  dict(cum_edges),
        })

    print(f'{len(snapshots)} time periods, '
          f'{len(cum_nodes)} contributors, {len(cum_edges)} collaborations total')
    return snapshots

## Compute layout

In [ ]:
def compute_layout(snapshots):
    """Spring layout on the final graph — core contributors cluster at the centre."""
    final = snapshots[-1]
    G = nx.Graph()
    G.add_nodes_from(final['nodes'])
    G.add_edges_from(final['edges'])
    pos = nx.spring_layout(G, k=2.0, iterations=120, seed=42)
    # Normalise to [-1, 1]
    coords = np.array(list(pos.values()))
    centre = coords.mean(axis=0)
    scale  = np.abs(coords - centre).max()
    return {n: tuple((np.array(xy) - centre) / (scale or 1)) for n, xy in pos.items()}

## Matplotlib animation (GIF / MP4)

In [ ]:
_BG   = '#0d1117'
_CMAP = plt.cm.plasma


def _draw_frame(ax, snapshot, pos, period_idx, n_periods, prev_nodes):
    ax.clear()
    ax.set_facecolor(_BG)
    ax.axis('off')
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)

    nodes    = snapshot['nodes']
    edges    = snapshot['edges']
    new_nodes = set(nodes) - prev_nodes

    # --- edges ---
    for (a, b), ed in edges.items():
        if a not in pos or b not in pos:
            continue
        age   = period_idx[snapshot['period']] - period_idx[ed['first_seen']]
        alpha = min(0.55, 0.08 + 0.06 * age)
        lw    = 0.3 + 0.12 * min(ed['weight'], 6)
        ax.plot([pos[a][0], pos[b][0]], [pos[a][1], pos[b][1]],
                '-', color='#58a6ff', alpha=alpha, linewidth=lw, zorder=1)

    # --- nodes ---
    top_nodes = {n for n, d in sorted(nodes.items(),
                 key=lambda x: x[1]['pr_count'], reverse=True)[:12]}
    for node, nd in nodes.items():
        if node not in pos:
            continue
        x, y    = pos[node]
        c_val   = period_idx[nd['first_seen']] / max(1, n_periods - 1)
        color   = _CMAP(c_val)
        size    = 25 + 8 * min(nd['pr_count'], 30)
        is_new  = node in new_nodes
        # glow ring
        ax.scatter(x, y, s=size * (5 if is_new else 3),
                   color=color, alpha=0.45 if is_new else 0.12, zorder=2)
        # core dot
        ax.scatter(x, y, s=size, color=color, alpha=0.95, zorder=3)
        # label for prominent contributors
        if node in top_nodes:
            ax.text(x, y + 0.09, node, ha='center', fontsize=5,
                    color='#e6edf3', alpha=0.85, zorder=4)

    # --- overlay ---
    ax.text(0.02, 0.97, str(snapshot['period']),
            transform=ax.transAxes, color='white', fontsize=14,
            va='top', fontweight='bold')
    ax.text(0.02, 0.91,
            f"{len(nodes)} contributors  •  {len(edges)} collaborations",
            transform=ax.transAxes, color='#8b949e', fontsize=8, va='top')


def make_animation(snapshots, pos, fps=4, output='collaboration_growth.gif'):
    period_idx = {s['period']: i for i, s in enumerate(snapshots)}
    n_periods  = len(snapshots)
    prev_list  = [set()] + [set(snapshots[i - 1]['nodes']) for i in range(1, n_periods)]

    fig, ax = plt.subplots(figsize=(10, 10), facecolor=_BG)
    fig.tight_layout(pad=0)

    def update(i):
        _draw_frame(ax, snapshots[i], pos, period_idx, n_periods, prev_list[i])

    anim = FuncAnimation(fig, update, frames=n_periods,
                         interval=1000 // fps, repeat=True)

    if output.endswith('.gif'):
        anim.save(output, writer=PillowWriter(fps=fps), dpi=120)
    else:
        from matplotlib.animation import FFMpegWriter
        anim.save(output, writer=FFMpegWriter(fps=fps, bitrate=2000), dpi=150)
    plt.close()
    print(f'Saved → {output}')
    return anim

## Plotly interactive animation

In [ ]:
def make_plotly_animation(snapshots, pos, repo=''):
    period_idx = {s['period']: i for i, s in enumerate(snapshots)}
    n_periods  = len(snapshots)
    cmap       = plt.cm.plasma

    def rgba(period):
        r, g, b, _ = cmap(period_idx[period] / max(1, n_periods - 1))
        return f'rgb({int(r*255)},{int(g*255)},{int(b*255)})'

    def snapshot_traces(snapshot):
        nodes, edges = snapshot['nodes'], snapshot['edges']
        ex, ey = [], []
        for (a, b) in edges:
            if a in pos and b in pos:
                ex += [pos[a][0], pos[b][0], None]
                ey += [pos[a][1], pos[b][1], None]
        present = [(n, d) for n, d in nodes.items() if n in pos]
        nx_c  = [pos[n][0] for n, _ in present]
        ny_c  = [pos[n][1] for n, _ in present]
        tips  = [f'{n}<br>since {d["first_seen"]}<br>{d["pr_count"]} PRs'
                 for n, d in present]
        colors = [rgba(d['first_seen']) for _, d in present]
        sizes  = [7 + 2 * min(d['pr_count'], 20) for _, d in present]
        return [
            go.Scatter(x=ex, y=ey, mode='lines',
                       line=dict(color='#4a90d9', width=0.6), opacity=0.35,
                       hoverinfo='skip'),
            go.Scatter(x=nx_c, y=ny_c, mode='markers',
                       text=[n for n, _ in present],
                       hovertext=tips, hoverinfo='text',
                       marker=dict(size=sizes, color=colors, opacity=0.92,
                                   line=dict(width=0))),
        ]

    frames = [go.Frame(name=str(s['period']), data=snapshot_traces(s))
              for s in snapshots]

    fig = go.Figure(
        data=frames[0].data,
        frames=frames,
        layout=go.Layout(
            paper_bgcolor=_BG, plot_bgcolor=_BG,
            showlegend=False,
            xaxis=dict(visible=False, range=[-1.4, 1.4]),
            yaxis=dict(visible=False, range=[-1.4, 1.4], scaleanchor='x'),
            margin=dict(l=10, r=10, t=50, b=60),
            title=dict(text=f'Collaboration Network Growth — {repo}',
                       font=dict(color='white', size=15)),
            updatemenus=[dict(
                type='buttons', showactive=False,
                y=-0.08, x=0.5, xanchor='center',
                buttons=[
                    dict(label='▶ Play', method='animate',
                         args=[None, dict(frame=dict(duration=700, redraw=True),
                                         fromcurrent=True)]),
                    dict(label='⏸ Pause', method='animate',
                         args=[[None], dict(frame=dict(duration=0),
                                            mode='immediate')]),
                ]
            )],
            sliders=[dict(
                currentvalue=dict(prefix='Period: ', font=dict(color='white')),
                font=dict(color='white'),
                pad=dict(t=10),
                steps=[dict(
                    args=[[str(s['period'])],
                          dict(frame=dict(duration=300), mode='immediate')],
                    label=str(s['period']),
                    method='animate',
                ) for s in snapshots],
            )],
        )
    )
    return fig

## Run

In [ ]:
HEADERS    = make_headers(GITHUB_TOKEN)
pr_records = fetch_pr_data(REPO, HEADERS, max_prs=MAX_PRS)

snapshots = build_snapshots(pr_records, freq=TIME_RESOLUTION)
pos       = compute_layout(snapshots)

# --- Movie ---
make_animation(snapshots, pos, fps=FPS, output=OUTPUT_FILE)

# Display GIF inline if in Jupyter
try:
    from IPython.display import Image, display
    if OUTPUT_FILE.endswith('.gif'):
        display(Image(OUTPUT_FILE))
except ImportError:
    pass

# --- Interactive Plotly ---
if HAS_PLOTLY:
    fig = make_plotly_animation(snapshots, pos, repo=REPO)
    fig.show()